<a href="https://colab.research.google.com/github/RomanLuja/ML/blob/main/03_data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Заняття 3. Якість даних і валідація для ML

**Блок 2. Дані як основа ML**

Ви вже вмієте чистити дані в `pandas`. Сьогодні — те, що **специфічне для машинного навчання**:
які проблеми даних ламають модель непомітно і як перевіряти дані до навчання.

## 1. Чек-лист «чи готові дані до ML»

| # | Перевірка | Чому це важливо для моделі |
|---|---|---|
| 1 | **Що таке один рядок?** Немає дублікатів | дублікати потрапляють і в train, і в test — оцінка завищена |
| 2 | **Ціль** є, без пропусків, зрозуміло як визначена | без `y` нема чого вчити |
| 3 | **Баланс класів** (для класифікації) | при 5 % «так» модель «завжди ні» має 95 % точності |
| 4 | **Неможливі значення** (вік 0, від'ємний дохід) | модель вчиться на помилках введення |
| 5 | **Неузгоджені категорії** (`Kyiv`, `kyiv `, `KYIV`) | модель бачить три різні міста |
| 6 | **Пропуски** | більшість моделей `scikit-learn` не приймають `NaN` |
| 7 | **Витік цілі (target leakage)** | ознака відома лише *після* події — модель «підглядає» відповідь |

### Витік цілі — найнебезпечніша помилка

Запитайте про кожну ознаку: **«Чи знатиму я це значення в момент, коли потрібен прогноз?»**

Приклад: прогнозуємо, чи купить лід (потенційний клієнт). Ознака «сума угоди» заповнена лише для
тих, хто купив. Модель з нею буде «ідеальною» на історії й марною в реальності: для нового ліда
суми угоди ще немає.

**Ознака витоку:** результат «занадто хороший, щоб бути правдою» (95–100 % там, де бізнес
очікує 70 %).

## 2. Демонстрація: ліди відділу продажів

B2B-компанія хоче прогнозувати, які ліди стануть клієнтами (`converted`), щоб менеджери
спершу дзвонили найперспективнішим.

In [1]:
# Підготовка: бібліотеки та функція завантаження даних (працює в Colab і локально)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
DATA_URL = "https://raw.githubusercontent.com/KsiuTretyakova/ml_for_analys_dta_2026/refs/heads/main/data/"


def load_csv(name):
    """Читає CSV з поточної теки, з теки data/, з DATA_URL або просить завантажити файл у Colab."""
    for path in (name, os.path.join("data", name)):
        if os.path.exists(path):
            return pd.read_csv(path)
    if DATA_URL:
        return pd.read_csv(DATA_URL.rstrip("/") + "/" + name)
    from google.colab import files
    print(f"Натисніть «Choose files» і виберіть файл {name}")
    files.upload()
    return pd.read_csv(name)

In [2]:
leads = load_csv("crm_leads.csv")
print(leads.shape)
leads.head()

(3090, 12)


,lead_id,created_at,lead_source,industry,employees,pages_visited,emails_opened,demo_requested,days_to_first_reply,deal_amount_uah,contract_signed_date,converted
0,L51232,2025-09-07,Referral,Finance,17,5,1,1,NaN,44700.0,2025-10-06,1
1,L50548,2025-01-15,Website,Manufacturing,25,9,2,0,0.0,NaN,NaN,0
2,L50523,2025-07-22,Event,Finance,15,5,3,0,1.6,NaN,NaN,0
3,L52786,2025-06-13,Website,Logistics,164,2,4,0,0.8,NaN,NaN,0
4,L52244,2025-09-22,Website,IT,44,4,4,0,2.0,NaN,NaN,0


### Перевірки 1–2: рядок, дублікати, ціль

In [3]:
print("Повних дублікатів:", leads.duplicated().sum())
print("Дублікатів lead_id:", leads["lead_id"].duplicated().sum())
print("Пропусків у цілі:", leads["converted"].isna().sum())
leads = leads.drop_duplicates().reset_index(drop=True)
print("Після видалення дублікатів:", leads.shape)

Повних дублікатів: 90
Дублікатів lead_id: 90
Пропусків у цілі: 0
Після видалення дублікатів: (3000, 12)


### Перевірка 3: баланс класів

In [4]:
leads["converted"].value_counts(normalize=True).round(3)

,proportion
converted,
0,0.637
1,0.363


Близько третини лідів купують. Отже, «модель», яка завжди каже «не купить», уже матиме приблизно
64 % правильних відповідей. Це наш **орієнтир**: модель має бути помітно кращою.

### Перевірки 4–6: неможливі значення, категорії, пропуски

In [5]:
leads.head()

,lead_id,created_at,lead_source,industry,employees,pages_visited,emails_opened,demo_requested,days_to_first_reply,deal_amount_uah,contract_signed_date,converted
0,L51232,2025-09-07,Referral,Finance,17,5,1,1,NaN,44700.0,2025-10-06,1
1,L50548,2025-01-15,Website,Manufacturing,25,9,2,0,0.0,NaN,NaN,0
2,L50523,2025-07-22,Event,Finance,15,5,3,0,1.6,NaN,NaN,0
3,L52786,2025-06-13,Website,Logistics,164,2,4,0,0.8,NaN,NaN,0
4,L52244,2025-09-22,Website,IT,44,4,4,0,2.0,NaN,NaN,0


In [6]:
leads.describe().round(1)

,employees,pages_visited,emails_opened,demo_requested,days_to_first_reply,deal_amount_uah,converted
count,3000.0,3000.0,3000.0,3000.0,2960.0,1088.0,3000.0
mean,3468.1,5.0,3.0,0.3,2.0,71646.2,0.4
std,57641.1,2.2,1.8,0.4,2.0,47494.1,0.5
min,-1.0,0.0,0.0,0.0,0.0,9400.0,0.0
25%,23.8,3.0,2.0,0.0,0.6,39875.0,0.0
50%,57.0,5.0,3.0,0.0,1.3,59300.0,0.0
75%,143.0,6.0,4.0,1.0,2.8,90200.0,1.0
max,999999.0,14.0,10.0,1.0,19.3,411000.0,1.0


In [7]:
print("Унікальні джерела до чистки:", leads["lead_source"].unique())
leads["lead_source"] = leads["lead_source"].str.strip().str.title()
print("Після чистки:", leads["lead_source"].unique())

bad_employees = (leads["employees"] <= 0) | (leads["employees"] > 100_000)
print("Неможлива кількість працівників:", bad_employees.sum())
leads.loc[bad_employees, "employees"] = np.nan   # помилку перетворюємо на пропуск

leads.isna().sum()

Унікальні джерела до чистки: ['Referral' 'Website' 'Event' 'website' 'Cold call' 'ads' 'Ads' ' Event '
 'cold call' 'event' ' Referral ' ' Cold call ' ' Website ' ' Ads '
 'referral']
Після чистки: ['Referral' 'Website' 'Event' 'Cold Call' 'Ads']
Неможлива кількість працівників: 35


,0
lead_id,0
created_at,0
lead_source,0
industry,150
employees,35
pages_visited,0
emails_opened,0
demo_requested,0
days_to_first_reply,40
deal_amount_uah,1912


`deal_amount_uah` і `contract_signed_date` мають пропуски приблизно в 64 % рядків — рівно в тих,
хто **не купив**. Це перша підказка про витік.

### Перевірка 7: витік цілі

Покажемо, що станеться, якщо витік не помітити. Створимо ознаку «є дата підписання договору» і
навчимо просту модель класифікації — **дерево рішень**.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

leads["has_contract_date"] = leads["contract_signed_date"].notna().astype(int)

leaky_features = ["pages_visited", "emails_opened", "demo_requested", "has_contract_date"]
X_tr, X_te, y_tr, y_te = train_test_split(leads[leaky_features], leads["converted"],
                                          test_size=0.25, random_state=42, stratify=leads["converted"])
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
print(f"Частка правильних відповідей з витоком: {tree.score(X_te, y_te):.1%}")

Частка правильних відповідей з витоком: 100.0%


100 % — «занадто добре, щоб бути правдою». Дата договору з'являється **після** покупки, тому для
нового ліда її не буде ніколи. Видаляємо обидві колонки-витоки.

## 3. Правильне розділення і пропуски без витоку

Пропуски заповнюємо **статистикою з навчальних даних** (наприклад, медіаною). Якщо порахувати
медіану по всьому датасету, тестові дані «просочаться» в навчання — ще один, тихіший витік.

Порядок завжди такий: **розділити → навчити заповнювач на train → застосувати до train і test**.

`stratify=y` зберігає однакову частку класів у train і test, `random_state=42` робить розділення
відтворюваним.

In [9]:
from sklearn.impute import SimpleImputer

features = ["employees", "pages_visited", "emails_opened", "demo_requested", "days_to_first_reply"]
X = leads[features]
y = leads["converted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

imputer = SimpleImputer(strategy="median")
X_tr_f = imputer.fit_transform(X_tr)   # fit: вчимо медіани ЛИШЕ на train
X_te_f = imputer.transform(X_te)       # transform: застосовуємо ті самі медіани до test
print("Медіани з train:", dict(zip(features, imputer.statistics_)))

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr_f, y_tr)
print(f"Частка правильних відповідей без витоку: {tree.score(X_te_f, y_te):.1%}")
print(f"Орієнтир «завжди ні»: {1 - y_te.mean():.1%}")

Медіани з train: {'employees': np.float64(58.0), 'pages_visited': np.float64(5.0), 'emails_opened': np.float64(3.0), 'demo_requested': np.float64(0.0), 'days_to_first_reply': np.float64(1.3)}
Частка правильних відповідей без витоку: 70.9%
Орієнтир «завжди ні»: 63.7%


Чесна модель помітно краща за орієнтир, але далеко не ідеальна — так і виглядає реальність.
Тепер цю модель можна застосувати до нового ліда.

## 4. Валідація даних як код

Нові дані приходитимуть щотижня. Замість того щоб щоразу перевіряти очима, напишемо функцію, яка
повертає список проблем. Порожній список — дані готові.

In [10]:
def validate_leads(df):
    problems = []
    if df.duplicated().any():
        problems.append(f"дублікати рядків: {df.duplicated().sum()}")
    if df["converted"].isna().any():
        problems.append("пропуски в цілі converted")
    if not df["converted"].isin([0, 1]).all():
        problems.append("ціль має значення, крім 0 і 1")
    bad = ((df["employees"] <= 0) | (df["employees"] > 100_000)).sum()
    if bad:
        problems.append(f"неможлива кількість працівників: {bad}")
    allowed_sources = {"Website", "Referral", "Ads", "Event", "Cold Call"}
    unknown = set(df["lead_source"].dropna()) - allowed_sources
    if unknown:
        problems.append(f"невідомі джерела: {unknown}")
    for col in ["deal_amount_uah", "contract_signed_date"]:
        if col in df.columns:
            problems.append(f"колонка-витік {col} - видалити перед навчанням")
    return problems

print("Сирі дані:")
for p in validate_leads(load_csv("crm_leads.csv")):
    print(" -", p)

clean = leads.drop(columns=["deal_amount_uah", "contract_signed_date", "has_contract_date"])
print("Очищені дані:", validate_leads(clean))

Сирі дані:
 - дублікати рядків: 90
 - неможлива кількість працівників: 35
 - невідомі джерела: {'ads', ' Cold call ', 'Cold call', ' Event ', 'website', ' Referral ', 'cold call', 'event', ' Website ', 'referral', ' Ads '}
 - колонка-витік deal_amount_uah - видалити перед навчанням
 - колонка-витік contract_signed_date - видалити перед навчанням
Очищені дані: []


---
## 5. Практика: кредитні заявки банку

Банк хоче прогнозувати дефолт за кредитом (`default`: 1 — не повернув). Файл
`bank_loans_raw.csv` — сире вивантаження з бази. Пройдіть увесь чек-лист.

### Завдання 1. Перевірки 1–3

Завантажте дані, знайдіть і видаліть дублікати, перевірте ціль і баланс класів. Яку частку
правильних відповідей має «модель», що завжди каже «поверне»?

In [11]:
# Завантажте дані

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
DATA_URL = "https://raw.githubusercontent.com/KsiuTretyakova/ml_for_analys_dta_2026/refs/heads/main/data/"

def load_csv(name):
    """Читає CSV з поточної теки, з теки data/, з DATA_URL або просить завантажити файл у Colab."""
    for path in (name, os.path.join("data", name)):
        if os.path.exists(path):
            return pd.read_csv(path)
    if DATA_URL:
        return pd.read_csv(DATA_URL.rstrip("/") + "/" + name)
    from google.colab import files
    print(f"Натисніть «Choose files» і виберіть файл {name}")
    files.upload()
    return pd.read_csv(name)

# Завантаження даних
loans = load_csv("bank_loans_raw.csv")
print("Розмір до очищення:", loans.shape)
loans.head()



Розмір до очищення: (5070, 15)


,client_id,age,employment_type,monthly_income_uah,years_employed,loan_amount_uah,term_months,interest_rate,credit_history_years,prior_defaults,open_loans,region,has_collateral,default,collection_calls
0,K203162,58,employed,47600.0,6.0,186000.0,6,23.2,2.8,0,0,Odesa,0,0,0
1,K200019,53,employed,16700.0,1.1,38000.0,36,16.1,3.4,0,2,Lviv,0,0,0
2,K201618,20,retired,19100.0,0.0,220000.0,12,32.1,2.0,0,0,Dnipro,0,0,0
3,K201585,50,self_employed,12400.0,19.8,61000.0,24,23.8,3.5,1,1,Other,0,0,0
4,K204553,62,employed,30500.0,1.7,48000.0,24,30.1,3.2,0,0,Other,0,0,0


In [12]:
# Перевірки 1–2: рядок, дублікати, ціль

print("Повних дублікатів:", loans.duplicated().sum())
print("Дублікатів lead_id:", loans["client_id"].duplicated().sum())
print("Пропусків у цілі:", loans["default"].isna().sum())
loans = loans.drop_duplicates().reset_index(drop=True)
print("Після видалення дублікатів:", loans.shape)

Повних дублікатів: 70
Дублікатів lead_id: 70
Пропусків у цілі: 0
Після видалення дублікатів: (5000, 15)


In [13]:
# Перевірка 3: баланс класів

loans["default"].value_counts(normalize=True).round(3)

,proportion
default,
0,0.835
1,0.165


In [14]:
# Яку частку правильних відповідей має «модель», що завжди каже «поверне»?


### Завдання 2. Перевірки 4–6

1. Знайдіть неможливі значення віку та доходу й замініть їх на `np.nan`.
2. Приведіть `region` до єдиного вигляду.
3. Виведіть кількість пропусків у кожній колонці.

In [15]:
# Ваш код тут
loans.describe().round(1)

,age,monthly_income_uah,years_employed,loan_amount_uah,term_months,interest_rate,credit_history_years,prior_defaults,open_loans,has_collateral,default,collection_calls
count,5000.0,4880.0,4800.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0
mean,40.2,21861.6,4.7,139324.0,27.4,27.9,5.6,0.3,1.2,0.3,0.2,0.8
std,29.8,12346.8,4.1,109821.1,16.4,6.1,3.9,0.7,1.1,0.5,0.4,1.7
min,0.0,-48500.0,0.0,5000.0,6.0,9.0,0.1,0.0,0.0,0.0,0.0,0.0
25%,32.0,13600.0,1.4,67000.0,12.0,23.9,2.7,0.0,0.0,0.0,0.0,0.0
50%,39.0,20000.0,3.9,110000.0,24.0,27.9,4.8,0.0,1.0,0.0,0.0,0.0
75%,46.0,28000.0,6.9,176250.0,36.0,32.0,7.6,0.0,2.0,1.0,0.0,0.0
max,999.0,99400.0,26.9,1374000.0,60.0,49.9,29.5,3.0,8.0,1.0,1.0,12.0


In [16]:
loans.head()

,client_id,age,employment_type,monthly_income_uah,years_employed,loan_amount_uah,term_months,interest_rate,credit_history_years,prior_defaults,open_loans,region,has_collateral,default,collection_calls
0,K203162,58,employed,47600.0,6.0,186000.0,6,23.2,2.8,0,0,Odesa,0,0,0
1,K200019,53,employed,16700.0,1.1,38000.0,36,16.1,3.4,0,2,Lviv,0,0,0
2,K201618,20,retired,19100.0,0.0,220000.0,12,32.1,2.0,0,0,Dnipro,0,0,0
3,K201585,50,self_employed,12400.0,19.8,61000.0,24,23.8,3.5,1,1,Other,0,0,0
4,K204553,62,employed,30500.0,1.7,48000.0,24,30.1,3.2,0,0,Other,0,0,0


In [17]:
# 1. Неможливі значення віку та доходу -> np.nan
bad_age = (loans["age"] < 18) | (loans["age"] > 100)
bad_income = loans["monthly_income_uah"] < 0
print("Неможливий вік:", bad_age.sum())
print("Неможливий дохід:", bad_income.sum())

loans.loc[bad_age, "age"] = np.nan
loans.loc[bad_income, "monthly_income_uah"] = np.nan

# 2. region до єдиного вигляду
print("До чистки:", loans["region"].unique())
loans["region"] = loans["region"].str.strip().str.title()
print("Після чистки:", loans["region"].unique())

# 3. Пропуски в кожній колонці
loans.isna().sum()


Неможливий вік: 30
Неможливий дохід: 25
До чистки: ['Odesa' 'Lviv' 'Dnipro' 'Other' 'Kyiv' 'Kharkiv' nan 'kyiv ' 'DNIPRO'
 'KYIV' 'other ' 'OTHER' 'ODESA' 'dnipro ' 'LVIV' 'lviv ' 'kharkiv '
 'KHARKIV' 'odesa ']
Після чистки: ['Odesa' 'Lviv' 'Dnipro' 'Other' 'Kyiv' 'Kharkiv' nan]


,0
client_id,0
age,30
employment_type,0
monthly_income_uah,145
years_employed,200
loan_amount_uah,0
term_months,0
interest_rate,0
credit_history_years,0
prior_defaults,0


### Завдання 3. Знайдіть витік

1. Порівняйте середні значення числових колонок для `default = 0` і `default = 1`
   (`groupby("default").mean(numeric_only=True)`). Яка ознака відрізняється підозріло сильно?
2. Чи буде вона відома в момент, коли клієнт подає заявку?
3. Доведіть витік: навчіть `DecisionTreeClassifier(max_depth=3, random_state=42)` лише на цій
   ознаці й подивіться на частку правильних відповідей.

In [18]:
# collection_calls - дзвінки колекторів - бувають лише ПІСЛЯ прострочення. Це витік.

from sklearn.impute import SimpleImputer

features = ["employees", "pages_visited", "emails_opened", "demo_requested", "days_to_first_reply"]
X = leads[features]
y = leads["converted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

imputer = SimpleImputer(strategy="median")
X_tr_f = imputer.fit_transform(X_tr)   # fit: вчимо медіани ЛИШЕ на train
X_te_f = imputer.transform(X_te)       # transform: застосовуємо ті самі медіани до test
print("Медіани з train:", dict(zip(features, imputer.statistics_)))

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr_f, y_tr)
print(f"Частка правильних відповідей без витоку: {tree.score(X_te_f, y_te):.1%}")
print(f"Орієнтир «завжди ні»: {1 - y_te.mean():.1%}")

Медіани з train: {'employees': np.float64(58.0), 'pages_visited': np.float64(5.0), 'emails_opened': np.float64(3.0), 'demo_requested': np.float64(0.0), 'days_to_first_reply': np.float64(1.3)}
Частка правильних відповідей без витоку: 70.9%
Орієнтир «завжди ні»: 63.7%


### Завдання 4. Чесна модель

1. Видаліть витік. Ознаки: `age`, `monthly_income_uah`, `years_employed`, `loan_amount_uah`,
   `term_months`, `interest_rate`, `credit_history_years`, `prior_defaults`, `open_loans`, `has_collateral`.
2. Розділіть на train/test (25 %, `random_state=42`, `stratify`).
3. Заповніть пропуски медіанами **з train**.
4. Навчіть дерево `max_depth=4` і порівняйте частку правильних відповідей з орієнтиром із завдання 1.

In [19]:
# Перевага над орієнтиром невелика: «частка правильних» погано підходить для
# незбалансованих класів. Кращі метрики для таких задач - на занятті 7.
# Ваш код тут

### Завдання 5. Функція валідації

Напишіть `validate_loans(df)`, яка повертає список проблем: дублікати, пропуски в цілі,
вік поза 18–100, дохід ≤ 0, регіони поза списком `Kyiv, Lviv, Kharkiv, Odesa, Dnipro, Other`,
наявність колонки `collection_calls`. Запустіть її на сирих даних і на очищеній таблиці.

In [20]:
# Ваш код тут

---
## Підсумок

- Перед навчанням — **чек-лист із 7 перевірок**. Найнебезпечніша помилка — **витік цілі**.
- Результат «занадто добрий» — привід шукати витік, а не святкувати.
- Порядок: **розділити → навчити обробку на train → застосувати до test**.
- Перевірки варто оформити функцією і запускати на кожному новому вивантаженні.